# 🟡 Solution: Line-Polygon Intersections

**Primitive:** vectorised Cramer's rule over all V edges

**Reduction:** `p = origin + t*dir = v1 + s*(v2-v1)` → solve for t and s via `det = cross(dir, edge)`; filter `s ∈ [0,1]`; sort by t.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import numpy as np

In [ ]:
# primitive: vectorised Cramer's rule (parametric line-segment intersection)

import numpy as np

def line_polygon_intersections(line_origin, line_dir, polygon):
    o    = np.asarray(line_origin, dtype=float)
    d    = np.asarray(line_dir,    dtype=float)
    poly = np.asarray(polygon,     dtype=float)
    v1 = poly; v2 = np.roll(poly, -1, axis=0)
    edge = v2 - v1; rhs = v1 - o           # (V, 2) each
    det = d[0]*edge[:,1] - d[1]*edge[:,0]  # (V,) cross(d, edge)
    par = np.abs(det) < 1e-10
    sd  = np.where(par, 1.0, det)
    t = (rhs[:,0]*edge[:,1] - rhs[:,1]*edge[:,0]) / sd
    s = (rhs[:,0]*d[1]     - rhs[:,1]*d[0])      / sd
    valid = ~par & (s >= 0.) & (s <= 1.)
    t_v = t[valid]
    if len(t_v) == 0:
        return np.zeros((0, 2))
    return o + t_v[np.argsort(t_v), None] * d

In [ ]:
# 🔍 Verify solution
sq = np.array([[0.,0.],[1.,0.],[1.,1.],[0.,1.]])

# Horizontal line through center
pts = line_polygon_intersections(np.array([0.5,0.5]), np.array([1.,0.]), sq)
print("Horizontal:", pts)          # expect [[0.,0.5],[1.,0.5]]

# Vertical line through center
pts2 = line_polygon_intersections(np.array([0.5,0.5]), np.array([0.,1.]), sq)
print("Vertical:", pts2)           # expect [[0.5,0.],[0.5,1.]]

In [ ]:
# ✅ Inline test suite
import numpy as np, time

sq = np.array([[0.,0.],[1.,0.],[1.,1.],[0.,1.]])

# ── Test 1: horizontal line through center ────────────────────────────────
r1 = line_polygon_intersections(np.array([0.5,0.5]), np.array([1.,0.]), sq)
r1 = np.array(r1, dtype=float)
assert r1.shape[0]==2, f"Expected 2 pts, got {r1.shape[0]}"
assert all(abs(r1[:,1]-0.5)<1e-9), f"y coords: {r1[:,1]}"
xs=sorted(r1[:,0])
assert abs(xs[0]-0.)<1e-9 and abs(xs[1]-1.)<1e-9, f"x coords: {xs}"
print("Test 1 passed: horizontal through center")

# ── Test 2: parallel — no intersection ────────────────────────────────────
r2 = line_polygon_intersections(np.array([0.,2.]), np.array([1.,0.]), sq)
r2 = np.array(r2)
assert r2.shape[0]==0, f"Parallel above: expected 0, got {r2.shape[0]}"
print("Test 2 passed: parallel line → no intersection")

# ── Test 3: diagonal line → sorted by t ───────────────────────────────────
sq2=np.array([[0.,0.],[2.,0.],[2.,2.],[0.,2.]])
r3=np.array(line_polygon_intersections(np.array([1.,1.]),np.array([1.,1.]),sq2),dtype=float)
assert r3.shape[0]==2, f"Diagonal: {r3.shape[0]}"
assert r3[0,0]<r3[1,0], f"Not sorted by t: {r3}"
print("Test 3 passed: diagonal → 2 points sorted by t")

# ── Test 4: line entirely outside ─────────────────────────────────────────
r4=np.array(line_polygon_intersections(np.array([5.,0.5]),np.array([0.,1.]),sq))
assert r4.shape[0]==0, f"Outside: expected 0, got {r4.shape[0]}"
print("Test 4 passed: line outside polygon")

# ── Test 5: large V=5000 ──────────────────────────────────────────────────
n=5000; a=np.linspace(0,2*np.pi,n,endpoint=False)
circle=np.stack([np.cos(a),np.sin(a)],axis=1)
t0=time.time()
r5=np.array(line_polygon_intersections(np.array([0.,0.]),np.array([1.,0.]),circle),dtype=float)
elapsed=time.time()-t0
assert r5.shape[0]==2, f"Circle: expected 2 pts, got {r5.shape[0]}"
assert elapsed<1.0, f"Too slow: {elapsed:.2f}s"
print(f"Test 5 passed: V=5000 ({elapsed:.3f}s)")

print("\nAll tests passed!")